In [4]:
%pip install -q msal requests pandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
USE_DEMO_DATA = True

# Fill in only after your Microsoft Entra app registration is approved.
CLIENT_ID = ""  # e.g. 12345678-1234-1234-1234-123456789abc
TENANT_ID = ""  # e.g. 12345678-1234-1234-1234-123456789abc

MAX_MESSAGES = 30  # Read only up to this many recent Inbox messages.
SCOPES = ["Mail.Read"]  # Delegated read permission; does not grant mail sending.

In [6]:
import msal
import requests
import pandas as pd
import re
from collections import Counter
from IPython.display import display

access_token = None

if USE_DEMO_DATA:
    print("DEMO MODE — no mailbox connection made.")
else:
    if not CLIENT_ID or not TENANT_ID:
        raise ValueError("Paste your CLIENT_ID and TENANT_ID in the configuration cell first.")

    app = msal.PublicClientApplication(
        CLIENT_ID,
        authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    )
    accounts = app.get_accounts()
    auth = app.acquire_token_silent(SCOPES, account=accounts[0]) if accounts else None
    if not auth:
        auth = app.acquire_token_interactive(scopes=SCOPES)

    if "access_token" not in auth:
        raise RuntimeError(
            "Microsoft sign-in failed: " +
            auth.get("error_description", auth.get("error", "Unknown error"))
        )
    access_token = auth["access_token"]
    print("Connected to Microsoft Graph. Access token kept in memory only.")

DEMO MODE — no mailbox connection made.


In [7]:
def fetch_inbox_messages(token, max_messages=30):
    url = "https://graph.microsoft.com/v1.0/me/mailFolders/inbox/messages"
    headers = {
        "Authorization": f"Bearer {token}",
        "Prefer": 'outlook.body-content-type="text"',
    }
    params = {
        "$top": min(max_messages, 50),
        "$select": "id,subject,from,receivedDateTime,body,hasAttachments",
        "$orderby": "receivedDateTime desc",
    }
    messages = []
    while url and len(messages) < max_messages:
        response = requests.get(url, headers=headers, params=params, timeout=30)
        if response.status_code in (401, 403):
            raise PermissionError(
                f"Graph returned HTTP {response.status_code}. "
                "Check your Microsoft 365 mailbox and delegated Mail.Read consent with IT."
            )
        response.raise_for_status()
        payload = response.json()
        messages.extend(payload.get("value", []))
        url = payload.get("@odata.nextLink")
        params = None  # nextLink already includes query parameters
    return messages[:max_messages]


def demo_email(sender, subject, body, received, attachments=False):
    return {
        "from": {"emailAddress": {"address": sender}},
        "subject": subject,
        "body": {"contentType": "text", "content": body},
        "receivedDateTime": received,
        "hasAttachments": attachments,
    }


demo_messages = [
    demo_email(
        "emma.fischer@example.com", "Purchase order – Apple",
        "Please BUY 150 shares of Apple, ISIN US0378331005, for account DEMO-001 "
        "at limit price USD 270.00. Thank you.", "2026-11-13T13:00:00Z"
    ),
    demo_email(
        "alexander.novak@example.com", "Sell Microsoft",
        "Please SELL 20 shares of Microsoft, ISIN US5949181045, "
        "account DEMO-002, at market.", "2026-11-13T12:50:00Z"
    ),
    demo_email(
        "lena.berger@example.com", "ETF order",
        "Please purchase 50 units of the ETF, ISIN IE00B4L5Y983, "
        "account DEMO-003, at market.", "2026-11-13T12:40:00Z"
    ),
    demo_email(
        "emil.weber@example.com", "Buy Apple",
        "Please BUY Apple shares (ISIN US0378331005) for account DEMO-004 "
        "at market. Please process today.", "2026-11-13T12:30:00Z"
    ),
    demo_email(
        "lea.schmidt@example.com", "Correction – Buy or sell?",
        "BUY 100 shares of Microsoft (ISIN US5949181045) for account DEMO-005 "
        "at market. Actually please SELL instead.", "2026-11-13T12:20:00Z"
    ),
    demo_email(
        "emma.fischer@example.com", "Purchase order – Apple (resend)",
        "Please BUY 150 shares of Apple, ISIN US0378331005, for account DEMO-001 "
        "at limit price USD 270.00. Thank you.", "2026-11-13T12:10:00Z"
    ),
    demo_email(
        "robert.klein@example.com", "Order in attached spreadsheet",
        "Please find our investment order in the attached Excel file for account DEMO-006.",
        "2026-11-13T12:00:00Z", attachments=True
    ),
]

messages = demo_messages if USE_DEMO_DATA else fetch_inbox_messages(access_token, MAX_MESSAGES)
print(f"Retrieved {len(messages)} messages from " + ("demo samples." if USE_DEMO_DATA else "your Inbox."))

Retrieved 7 messages from demo samples.


In [8]:
ISIN_PATTERN = re.compile(r"\b[A-Z]{2}[A-Z0-9]{9}\d\b")
QUANTITY_PATTERN = re.compile(
    r"\b(?:buy|purchase|sell|dispose(?:\s+of)?)\s+([\d,]+)\s+(?:shares?|units?)\b",
    re.IGNORECASE,
)
ACCOUNT_PATTERN = re.compile(
    r"\b(?:account|portfolio)(?:\s+(?:id|number|no\.?))?\s*[:#-]?\s*([A-Z0-9][A-Z0-9_-]{3,})\b",
    re.IGNORECASE,
)
LIMIT_PATTERN = re.compile(
    r"\blimit(?:\s+price)?\s*(?:of|at|:)?\s*(USD|EUR|CHF|GBP|\$|€|£)?\s*([\d,]+(?:\.\d+)?)",
    re.IGNORECASE,
)


def valid_isin(isin):
    """Validate the ISIN check digit using Luhn; does not verify the security."""
    if not ISIN_PATTERN.fullmatch(isin.upper()):
        return False
    expanded = "".join(
        str(ord(c) - 55) if c.isalpha() else c for c in isin.upper()
    )
    total = 0
    for i, character in enumerate(reversed(expanded)):
        d = int(character)
        if i % 2:
            d *= 2
            d = d // 10 + d % 10
        total += d
    return total % 10 == 0


def parse_order(message):
    sender = ((message.get("from") or {}).get("emailAddress") or {}).get("address", "")
    subject = message.get("subject") or ""
    body = (message.get("body") or {}).get("content") or ""
    text = subject + "\n" + body
    upper = text.upper()

    isins = sorted(set(ISIN_PATTERN.findall(upper)))
    directions = set()
    if re.search(r"\b(?:BUY|PURCHASE|ACQUIRE)\b", upper):
        directions.add("BUY")
    if re.search(r"\b(?:SELL|DISPOSE)\b", upper):
        directions.add("SELL")
    direction = next(iter(directions)) if len(directions) == 1 else None

    qty = QUANTITY_PATTERN.search(text)
    acct = ACCOUNT_PATTERN.search(text)
    limit_match = LIMIT_PATTERN.search(text)
    is_market = bool(re.search(r"\b(?:AT\s+MARKET|MARKET\s+ORDER)\b", upper))

    if limit_match and not is_market:
        order_type = "LIMIT"
    elif is_market and not limit_match:
        order_type = "MARKET"
    elif limit_match and is_market:
        order_type = "CONFLICTING"
    else:
        order_type = None

    exceptions = []
    if not isins:
        exceptions.append("Missing ISIN")
    elif len(isins) > 1:
        exceptions.append("Multiple ISINs; split/review")
    if any(not valid_isin(isin) for isin in isins):
        exceptions.append("Invalid ISIN checksum")
    if len(directions) > 1:
        exceptions.append("Conflicting BUY/SELL")
    elif not direction:
        exceptions.append("Missing BUY/SELL")
    if not qty:
        exceptions.append("Missing quantity or unsupported wording")
    if not acct:
        exceptions.append("Missing account")
    if order_type == "CONFLICTING":
        exceptions.append("Conflicting order type")
    elif not order_type:
        exceptions.append("Missing market/limit instruction")
    if message.get("hasAttachments"):
        exceptions.append("Attachment requires separate review")

    price = None
    if limit_match:
        price = ((limit_match.group(1) or "") + " " + limit_match.group(2)).strip()

    return {
        "Received": message.get("receivedDateTime", ""),
        "Sender": sender,
        "Subject": subject,
        "ISIN": ", ".join(isins),
        "ISIN checksum OK": bool(isins) and all(valid_isin(x) for x in isins),
        "Side": direction or ("CONFLICT" if len(directions) > 1 else ""),
        "Quantity": int(qty.group(1).replace(",", "")) if qty else None,
        "Account": acct.group(1) if acct else "",
        "Order type": order_type or "",
        "Limit price": price or "",
        "Exceptions": exceptions,
    }


rows = [parse_order(msg) for msg in messages]

# A same-sender, same-instruction resend may be a duplicate, not a new order.
seen = set()
for row in rows:
    fingerprint = (
        row["Sender"].casefold(), row["ISIN"], row["Side"], row["Quantity"],
        row["Account"], row["Order type"], row["Limit price"],
    )
    if row["ISIN"] and row["Quantity"] and fingerprint in seen:
        row["Exceptions"].append("Possible duplicate order")
    seen.add(fingerprint)
    row["Review status"] = (
        "Needs clarification" if row["Exceptions"] else "Pending manual controls"
    )
    row["Exceptions"] = "; ".join(row["Exceptions"])

results = pd.DataFrame(rows)
if results.empty:
    print("No messages were retrieved.")
else:
    print("Review status summary:")
    display(results["Review status"].value_counts().rename_axis("Status").to_frame("Emails"))
    display(results)

Review status summary:


,Emails
Status,
Needs clarification,4
Pending manual controls,3


,Received,Sender,Subject,ISIN,ISIN checksum OK,Side,Quantity,Account,Order type,Limit price,Exceptions,Review status
0,2026-11-13T13:00:00Z,emma.fischer@example.com,Purchase order – Apple,US0378331005,True,BUY,150.0,DEMO-001,LIMIT,USD 270.00,,Pending manual controls
1,2026-11-13T12:50:00Z,alexander.novak@example.com,Sell Microsoft,US5949181045,True,SELL,20.0,DEMO-002,MARKET,,,Pending manual controls
2,2026-11-13T12:40:00Z,lena.berger@example.com,ETF order,IE00B4L5Y983,True,BUY,50.0,DEMO-003,MARKET,,,Pending manual controls
3,2026-11-13T12:30:00Z,emil.weber@example.com,Buy Apple,US0378331005,True,BUY,NaN,DEMO-004,MARKET,,Missing quantity or unsupported wording,Needs clarification
4,2026-11-13T12:20:00Z,lea.schmidt@example.com,Correction – Buy or sell?,US5949181045,True,CONFLICT,100.0,DEMO-005,MARKET,,Conflicting BUY/SELL,Needs clarification
5,2026-11-13T12:10:00Z,emma.fischer@example.com,Purchase order – Apple (resend),US0378331005,True,BUY,150.0,DEMO-001,LIMIT,USD 270.00,Possible duplicate order,Needs clarification
6,2026-11-13T12:00:00Z,robert.klein@example.com,Order in attached spreadsheet,,False,,NaN,DEMO-006,,,Missing ISIN; Missing BUY/SELL; Missing quanti...,Needs clarification
